##⭐ Gold Layer Architecture (Important Concept)

⭐ Gold Layer Architecture (Important Concept)

- DIMENSION TABLES (descriptive)
- 
- customers
- products
- sellers
- geolocation
- 
- to
- 
- gold.dim_customers
- gold.dim_products
- gold.dim_sellers


⭐ FACT TABLES (transactions)

Main event = Order purchase

- orders
- order_items
- order_payments
- order_reviews
- 

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS e_comm_databricks.gold;

In [0]:
df_customers = spark.table("e_comm_databricks.silver.customers")

In [0]:
df_customers.write \
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("e_comm_databricks.gold.dim_customers")

In [0]:
df_products = spark.table("e_comm_databricks.silver.products")

In [0]:
df_products.write \
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("e_comm_databricks.gold.dim_products")

In [0]:
df_sellers = spark.table("e_comm_databricks.silver.sellers")

In [0]:
df_sellers.write \
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("e_comm_databricks.gold.dim_sellers")

Build Fact Table (MOST IMPORTANT)

We will create:

👉 fact_sales

This combines:
- orders
- JOIN order_items
- JOIN payments


In [0]:
orders = spark.table("e_comm_databricks.silver.orders").alias("o")
items = spark.table("e_comm_databricks.silver.order_items").alias("i")
payments = spark.table("e_comm_databricks.silver.order_payments").alias("p")


In [0]:
fact = orders.join(items, "order_id", "left") \
             .join(payments, "order_id", "left")


🧠 Why left join?

Because:

orders is main table

some orders may not have payments.

In [0]:
fact_clean = fact.select(

    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",

    "product_id",
    "seller_id",
    "price",

    "payment_type",
    "payment_value"

)


In [0]:
fact_clean.write \
 .format("delta") \
 .mode("overwrite") \
 .saveAsTable("e_comm_databricks.gold.fact_sales")


          Dimension tables
                |
                |
Dim_Product --- Fact_Table --- Dim_Customer
                |
                |
           Dim_Seller


In [0]:
%sql
select * from e_comm_databricks.gold.fact_sales

In [0]:
items = spark.table("e_comm_databricks.silver.order_items")
orders = spark.table("e_comm_databricks.silver.orders")

fact = items.join(
    orders,
    "order_id",
    "left"
)


In [0]:
fact_clean = fact.select(
    "order_id",
    "order_item_id",
    "product_id",
    "seller_id",
    "price",
    "freight_value",
    "customer_id",
    "order_status",
    "order_purchase_timestamp"
)


In [0]:
fact_clean.write \
 .format("delta") \
 .mode("overwrite") \
 .saveAsTable("e_comm_databricks.gold.fact_order_items")


In [0]:
payments = spark.table("e_comm_databricks.silver.order_payments")


In [0]:
fact_payments = payments.select(

    "order_id",
    "payment_sequential",
    "payment_type",
    "payment_installments",
    "payment_value"

)


In [0]:
fact_payments.write \
 .format("delta") \
 .mode("overwrite") \
 .saveAsTable("e_comm_databricks.gold.fact_payments")
